In [46]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'
)

df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [47]:
# Question 1
df = df[
    [
        'engine_displacement',
        'horsepower',
        'vehicle_weight',
        'model_year',
        'fuel_efficiency_mpg'
    ]
]

df.isnull().sum()

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

In [48]:
# Question 2
df['horsepower'].median()

np.float64(254.0)

In [49]:
#Question 3
# Split the data into train, validation and test sets
n = len(df)

n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)

idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

# Separate features and target
X_train = df_train.drop('fuel_efficiency_mpg', axis=1)
y_train = df_train['fuel_efficiency_mpg']

X_val = df_val.drop('fuel_efficiency_mpg', axis=1)
y_val = df_val['fuel_efficiency_mpg']


# Fill missing values with 0
X_train_zero = X_train.fillna(0)
X_val_zero = X_val.fillna(0)


# Train the linear regression model
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTy = X.T.dot(y)

    w = np.linalg.solve(XTX, XTy)

    return w[0], w[1:]


# Calculate RMSE
def rmse(y, y_pred):
    error = y_pred - y
    mse = (error ** 2).mean()
    return np.sqrt(mse)


# Train using 0 for missing values
w_0, w = train_linear_regression(X_train_zero, y_train)

y_pred = w_0 + X_val_zero.dot(w)

score_zero = rmse(y_val, y_pred)

print("RMSE with 0:", round(score_zero, 3))


# Fill missing values with the mean
mean = X_train.mean()

X_train_mean = X_train.fillna(mean)
X_val_mean = X_val.fillna(mean)


# Train using the mean for missing values
w_0, w = train_linear_regression(X_train_mean, y_train)

y_pred = w_0 + X_val_mean.dot(w)

score_mean = rmse(y_val, y_pred)

print("RMSE with mean:", round(score_mean, 3))

RMSE with 0: 2.205
RMSE with mean: 2.202


In [50]:
#Question 4
# Train linear regression with regularization
def train_linear_regression_reg(X, y, r):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)

    # Add regularization
    XTX = XTX + r * np.eye(XTX.shape[0])

    XTy = X.T.dot(y)

    w = np.linalg.solve(XTX, XTy)

    return w[0], w[1:]


# Prepare the training and validation data
X_train = df_train.drop('fuel_efficiency_mpg', axis=1).fillna(0)
X_val = df_val.drop('fuel_efficiency_mpg', axis=1).fillna(0)

y_train = df_train['fuel_efficiency_mpg']
y_val = df_val['fuel_efficiency_mpg']


# Try different regularization values
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    
    w_0, w = train_linear_regression_reg(X_train, y_train, r)

    y_pred = w_0 + X_val.dot(w)

    score = rmse(y_val, y_pred)

    print(r, round(score, 4))

0 2.2053
0.01 2.2058
0.1 2.2241
1 2.3492
5 2.4094
10 2.4195
100 2.4292


In [51]:
# Question 5
scores = []

# Try 10 different random seeds
for seed in range(10):

    np.random.seed(seed)

    # Shuffle the data
    idx = np.arange(len(df))
    np.random.shuffle(idx)

    # Split into train and validation sets
    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[idx[n_train:n_train + n_val]]

    # Prepare the training data
    X_train = df_train.drop('fuel_efficiency_mpg', axis=1).fillna(0)
    y_train = df_train['fuel_efficiency_mpg']

    # Prepare the validation data
    X_val = df_val.drop('fuel_efficiency_mpg', axis=1).fillna(0)
    y_val = df_val['fuel_efficiency_mpg']

    # Train the model
    w_0, w = train_linear_regression(X_train, y_train)

    # Make predictions
    y_pred = w_0 + X_val.dot(w)

    # Calculate RMSE
    score = rmse(y_val, y_pred)

    scores.append(score)

# Show all RMSE scores
print(scores)

# Calculate the standard deviation
print(round(np.std(scores), 3))

[np.float64(2.2392041430508307), np.float64(2.2021128210837753), np.float64(2.1634197787531178), np.float64(2.2043588768529827), np.float64(2.201880094331822), np.float64(2.245785150908247), np.float64(2.2697212079516507), np.float64(2.1907945999354297), np.float64(2.216611201694413), np.float64(2.2070365928232976)]
0.029


In [52]:
# Question 6
# Use seed 9
np.random.seed(9)

# Shuffle the data
idx = np.arange(len(df))
np.random.shuffle(idx)

# Split the data
df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

# Combine train and validation data
df_full_train = pd.concat([df_train, df_val])

# Prepare the training data
X_full_train = df_full_train.drop(
    'fuel_efficiency_mpg',
    axis=1
).fillna(0)

y_full_train = df_full_train['fuel_efficiency_mpg']

# Prepare the test data
X_test = df_test.drop(
    'fuel_efficiency_mpg',
    axis=1
).fillna(0)

y_test = df_test['fuel_efficiency_mpg']

# Train the model with regularization
w_0, w = train_linear_regression_reg(
    X_full_train,
    y_full_train,
    r=0.001
)

# Make predictions
y_pred = w_0 + X_test.dot(w)

# Calculate RMSE
score = rmse(y_test, y_pred)

print(round(score, 3))

2.236
